# 03 — Survey-calibrated cost model: does CNG pay for the drivers we surveyed?

**Question:** for the CNG drivers in the Lagos driver survey (S47), does the fuel saving outweigh the earnings
they lose queuing for gas, and what would change if they had paid the market price for their conversion?

Notebook 02 ran the model on published figures and assumptions. Here we plug in each surveyed driver's own reported
earnings, fuel spend, hours and queue times instead.

**Data:** `data/processed/survey_clean.csv`, made by `src/clean_survey.py`. Method and limitations:
`docs/survey_method.md`. Model functions: `src/cost_model.py`. Model assumptions and loan terms:
`data/raw/model_inputs.csv` (sources in `docs/sources.md`).

## 1. Setup

Same setup as notebook 02: find the project root whether Jupyter was started in `notebooks/` or the project folder,
then add `src/` to the import path so we can import our own modules.

In [1]:
import sys
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
import load_data          # noqa: E402
import cost_model as cm   # noqa: E402

FIG_DIR = PROJECT_ROOT / "outputs" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Show money without scientific notation in tables
pd.options.display.float_format = "{:,.2f}".format

## 2. Load the survey: unflagged CNG respondents only

We keep rows with an empty `quality_flag` (implausible money values are flagged by `clean_survey.py`, see
`docs/survey_method.md`) and with `uses_cng == True`, meaning CNG-only or bi-fuel vehicles.

pandas reads an empty cell as `NaN` (missing), not `""`, so "unflagged" is tested with `.isna()`.

In [2]:
survey = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "survey_clean.csv")

unflagged = survey[survey["quality_flag"].isna()]
cng = unflagged[unflagged["uses_cng"]].copy()

print(f"All responses: {len(survey)} | unflagged: {len(unflagged)} | unflagged CNG users: {len(cng)}")
print(cng["vehicle_type"].value_counts().to_string())

All responses: 25 | unflagged: 20 | unflagged CNG users: 13
vehicle_type
ride_hailing    7
danfo           5
keke            1


## 3. Per-driver calculations (as converted: loan cost = 0)

For each CNG driver:

| Quantity | Formula | Function in `cost_model.py` |
|---|---|---|
| Net earnings per hour | (gross per day − CNG spend per day) ÷ hours per day | — |
| Fuel saving | petrol spend before conversion − CNG spend now | — (reported directly, so no price conversion needed) |
| Queue cost | daily queue hours × net earnings per hour | `queue_cost_per_day` |
| Net benefit | fuel saving − queue cost − loan cost | `net_benefit_per_day` |
| Break-even daily queue hours | fuel saving ÷ net earnings per hour | `break_even_queue_hours` × refuels |

**Why use the model functions** when the formulas are this simple: it guarantees the survey results and the
scenario results in notebook 02 are calculated in exactly the same way.

`break_even_queue_hours` returns hours **per refuel**; multiplying by refuels per day gives hours **per day**, which
equals fuel saving ÷ net per hour.

**Loan cost is 0** here: 12 of the 13 drivers converted through the free government programme and the other's
owner paid (`payment_method`), and every reported conversion cost is 0 or blank.

**Note on "net":** this is gross minus fuel only. It does not deduct owner remittance or app commission, which the
survey did not ask about. Real take-home per hour is lower, so these queue costs are an upper bound.

In [3]:
cng["net_per_hour"] = (cng["gross_per_day"] - cng["cng_spend_per_day"]) / cng["hours_per_day"]
cng["fuel_saving"] = cng["petrol_before_per_day"] - cng["cng_spend_per_day"]

# The cost_model functions take single numbers, so apply them row by row.
# A list comprehension over zip(...) walks the columns side by side.
cng["queue_cost"] = [
    cm.queue_cost_per_day(refuels, hours, eph)
    for refuels, hours, eph in zip(cng["refuels_per_day"], cng["queue_hours"], cng["net_per_hour"])
]
cng["net_benefit"] = [
    cm.net_benefit_per_day(saving, queue, loan_cost=0)
    for saving, queue in zip(cng["fuel_saving"], cng["queue_cost"])
]
cng["break_even_daily_queue_hours"] = [
    cm.break_even_queue_hours(saving, 0, refuels, eph) * refuels
    for saving, refuels, eph in zip(cng["fuel_saving"], cng["refuels_per_day"], cng["net_per_hour"])
]

# Sanity check: the cost_model results match the plain formulas in the brief.
assert ((cng["queue_cost"] - cng["daily_queue_hours"] * cng["net_per_hour"]).abs() < 1e-6).all()
assert ((cng["break_even_daily_queue_hours"] - cng["fuel_saving"] / cng["net_per_hour"]).abs() < 1e-6).all()

per_driver_cols = [
    "respondent_id", "vehicle_type", "gross_per_day", "hours_per_day", "cng_spend_per_day",
    "petrol_before_per_day", "net_per_hour", "fuel_saving", "daily_queue_hours", "queue_cost",
    "net_benefit", "break_even_daily_queue_hours",
]
per_driver = cng[per_driver_cols].set_index("respondent_id")
per_driver.round(0).astype({"daily_queue_hours": float, "break_even_daily_queue_hours": float}).style.format(
    {c: "{:,.0f}" for c in per_driver.columns if c not in ("vehicle_type", "daily_queue_hours", "break_even_daily_queue_hours")}
    | {"daily_queue_hours": "{:.2f}", "break_even_daily_queue_hours": "{:.1f}"}
)

,vehicle_type,gross_per_day,hours_per_day,cng_spend_per_day,petrol_before_per_day,net_per_hour,fuel_saving,daily_queue_hours,queue_cost,net_benefit,break_even_daily_queue_hours
respondent_id,,,,,,,,,,,
R01,danfo,"80,000",7,"7,500","35,000","10,357","27,500",3.00,"31,071","-3,571",3.0
R02,ride_hailing,"120,000",8,"8,400","46,000","13,950","37,600",2.00,"20,925","16,675",3.0
R03,danfo,"90,000",11,"8,600","37,000","7,400","28,400",3.00,"22,200","6,200",4.0
R04,danfo,"140,000",10,"6,500","45,000","13,350","38,500",2.00,"30,038","8,462",3.0
R05,ride_hailing,"78,000",9,"8,200","60,000","7,756","51,800",1.00,"5,817","45,983",7.0
R06,ride_hailing,"180,000",9,"7,000","45,000","19,222","38,000",0.00,"9,611","28,389",2.0
R07,ride_hailing,"200,000",8,"9,000","45,000","23,875","36,000",2.00,"53,719","-17,719",2.0
R08,ride_hailing,"170,000",8,"8,000","67,000","20,250","59,000",2.00,"30,375","28,625",3.0
R10,danfo,"90,000",10,"4,200","35,000","8,580","30,800",1.00,"6,435","24,365",4.0


### Medians by vehicle type

Medians rather than means: with only 1–7 drivers per group, a single unusual answer would pull a mean a long way.
The keke group is **one driver**, so its "median" is just that driver.

In [4]:
median_cols = ["net_per_hour", "fuel_saving", "daily_queue_hours", "queue_cost", "net_benefit",
               "break_even_daily_queue_hours"]
by_vehicle = cng.groupby("vehicle_type")[median_cols].median()
by_vehicle.insert(0, "n", cng.groupby("vehicle_type").size())
by_vehicle.loc["all CNG users"] = [len(cng)] + list(cng[median_cols].median())
by_vehicle.style.format({"n": "{:.0f}", "net_per_hour": "{:,.0f}", "fuel_saving": "{:,.0f}",
                         "daily_queue_hours": "{:.2f}", "queue_cost": "{:,.0f}", "net_benefit": "{:,.0f}",
                         "break_even_daily_queue_hours": "{:.1f}"})

,n,net_per_hour,fuel_saving,daily_queue_hours,queue_cost,net_benefit,break_even_daily_queue_hours
vehicle_type,,,,,,,
danfo,5,"10,357","28,400",3.00,"30,038","6,200",2.9
keke,1,"4,475","5,800",3.00,"13,425","-7,625",1.3
ride_hailing,7,"13,950","38,000",1.50,"16,823","28,389",2.9
all CNG users,13,"11,215","36,000",2.25,"20,925","16,675",2.9


## 4. Market-price scenario: what if they had paid for the kit?

Same drivers, same fuel and queue figures, but now each one repays a **₦1,300,000** conversion on the loan terms in
`model_inputs.csv`: **17.5% a year over 2 years** (`loan_interest_rate` and `loan_tenor` mid, S18). ₦1.3m is the
`conversion_cost` car mid value (S40).

The loan repayment is spread over the days each driver **actually works**: `days_per_week × 52` from the survey
(312 or 364), instead of the model's assumed 312. Working more days means a smaller repayment per day.

**Caveat:** ₦1.3m is a car price, applied here to all vehicles as specified. For the one keke driver it overstates
the cost (the keke mid in `model_inputs.csv` is ₦375,000, Low confidence). No danfo conversion price is in the CSV.
The repayment applies only for the 2-year loan; after that the driver is back to the free-conversion figures above.

In [5]:
inputs = load_data.read_csv_safe(load_data.RAW_DIR / "model_inputs.csv")


def get_mid(parameter, vehicle):
    # Same idea as get_input in notebook 02, but returns just the mid value.
    match = inputs[(inputs["parameter"] == parameter) & (inputs["vehicle"] == vehicle)]
    if len(match) != 1:
        raise ValueError(f"Expected 1 row for {parameter} / {vehicle}, found {len(match)}")
    return float(match.iloc[0]["mid"])


market_conversion = get_mid("conversion_cost", "car")           # NGN, S40
interest_rate = get_mid("loan_interest_rate", "all") / 100      # CSV stores 17.5 (percent); function wants 0.175
tenor_years = get_mid("loan_tenor", "all")                      # years, S18
assert market_conversion == 1_300_000                           # the figure in the brief

cng["working_days_per_year"] = cng["days_per_week"] * 52
cng["loan_cost_market"] = [
    cm.loan_cost_per_day(market_conversion, interest_rate, tenor_years, days)
    for days in cng["working_days_per_year"]
]
cng["net_benefit_market"] = [
    cm.net_benefit_per_day(saving, queue, loan)
    for saving, queue, loan in zip(cng["fuel_saving"], cng["queue_cost"], cng["loan_cost_market"])
]
cng["break_even_daily_queue_hours_market"] = [
    cm.break_even_queue_hours(saving, loan, refuels, eph) * refuels
    for saving, loan, refuels, eph in zip(cng["fuel_saving"], cng["loan_cost_market"],
                                          cng["refuels_per_day"], cng["net_per_hour"])
]

print(f"Loan: NGN {market_conversion:,.0f} at {interest_rate:.1%} over {tenor_years:g} years")
market_cols = ["vehicle_type", "working_days_per_year", "net_benefit", "loan_cost_market", "net_benefit_market",
               "daily_queue_hours", "break_even_daily_queue_hours_market"]
cng.set_index("respondent_id")[market_cols].style.format(
    {"working_days_per_year": "{:.0f}", "net_benefit": "{:,.0f}", "loan_cost_market": "{:,.0f}",
     "net_benefit_market": "{:,.0f}", "daily_queue_hours": "{:.2f}",
     "break_even_daily_queue_hours_market": "{:.1f}"}
)

Loan: NGN 1,300,000 at 17.5% over 2 years


,vehicle_type,working_days_per_year,net_benefit,loan_cost_market,net_benefit_market,daily_queue_hours,break_even_daily_queue_hours_market
respondent_id,,,,,,,
R01,danfo,364,"-3,571","2,129","-5,701",3.00,2.4
R02,ride_hailing,312,"16,675","2,484","14,191",1.50,2.5
R03,danfo,364,"6,200","2,129","4,071",3.00,3.6
R04,danfo,312,"8,462","2,484","5,978",2.25,2.7
R05,ride_hailing,312,"45,983","2,484","43,499",0.75,6.4
R06,ride_hailing,312,"28,389","2,484","25,905",0.50,1.8
R07,ride_hailing,364,"-17,719","2,129","-19,848",2.25,1.4
R08,ride_hailing,364,"28,625","2,129","26,496",1.50,2.8
R10,danfo,364,"24,365","2,129","22,236",0.75,3.3


In [6]:
pays_free = (cng["net_benefit"] > 0).sum()
pays_market = (cng["net_benefit_market"] > 0).sum()
print(f"CNG pays (net benefit > 0): {pays_free} of {len(cng)} as converted (free), "
      f"{pays_market} of {len(cng)} at market price on a loan")
print(f"Median net benefit: NGN {cng['net_benefit'].median():,.0f}/day free, "
      f"NGN {cng['net_benefit_market'].median():,.0f}/day at market price")

CNG pays (net benefit > 0): 9 of 13 as converted (free), 9 of 13 at market price on a loan
Median net benefit: NGN 16,675/day free, NGN 14,191/day at market price


## 5. Survey vs model assumptions

The model column shows the **central** values notebook 02 used.

- **Earnings per hour:** the model has no single value, only a ₦1,000–5,000/hour sensitivity range (the worked
  example in `cost_model.py` uses ₦3,000). The survey figure is gross minus fuel, *before* owner remittance or app
  commission, so it is not directly comparable to the model's "net of remittance and commission".
- **Fuel saving %:** the model's figure is 1 − (CNG price ÷ petrol price) at the central prices, which is what
  `fuel_saving_per_day` gives with an efficiency ratio of 1. The survey figure is 1 − (CNG spend ÷ recalled petrol
  spend before conversion).

In [7]:
petrol_mid = get_mid("petrol_price_national", "all")   # NGN/L, S11
cng_mid = get_mid("cng_price", "car/bus")               # NGN/scm, S11
queue_mid = get_mid("queue_time_per_refuel", "all")     # hours, S02; S04
refuels_model = get_mid("refuels_per_day", "all")       # Assumption
days_model = get_mid("working_days_per_year", "all")    # Assumption
# Fuel saving share: run the model on a spend of 1 naira, so the saving is the share.
saving_share_model = cm.fuel_saving_per_day(1, petrol_mid, cng_mid, get_mid("efficiency_ratio", "all"))

survey_days = cng["days_per_week"] * 52
comparison = pd.DataFrame(
    [
        ("Earnings per hour (NGN)", f"{cng['net_per_hour'].median():,.0f}",
         "1,000–5,000 (range)", "Survey: gross minus CNG spend, per hour. Model: sensitivity range, no CSV row"),
        ("Refuels per day", f"{cng['refuels_per_day'].median():g}",
         f"{refuels_model:g}", "Model: Assumption"),
        ("Queue hours per refuel", f"{cng['queue_hours'].median():g}",
         f"{queue_mid:g}", "Survey: band midpoints. Model: S02; S04"),
        ("Daily queue hours", f"{cng['daily_queue_hours'].median():.2f}",
         f"{queue_mid * refuels_model:g}", "Model: queue per refuel x refuels per day"),
        ("Working days per year", f"{survey_days.median():.0f}",
         f"{days_model:.0f}", "Survey: days_per_week x 52. Model: Assumption"),
        ("Fuel saving %", f"{cng['fuel_saving_pct'].median():.1%}",
         f"{saving_share_model:.1%}", f"Model: 1 - {cng_mid:g}/{petrol_mid:,.0f} (S11)"),
    ],
    columns=["measure", f"survey median (n={len(cng)})", "model (central)", "note"],
).set_index("measure")
comparison

,survey median (n=13),model (central),note
measure,,,
Earnings per hour (NGN),"11,215","1,000–5,000 (range)","Survey: gross minus CNG spend, per hour. Model..."
Refuels per day,2,1,Model: Assumption
Queue hours per refuel,0.75,4,Survey: band midpoints. Model: S02; S04
Daily queue hours,2.25,4,Model: queue per refuel x refuels per day
Working days per year,364,312,Survey: days_per_week x 52. Model: Assumption
Fuel saving %,81.7%,77.3%,"Model: 1 - 318/1,400 (S11)"


## 6. Chart: actual vs break-even daily queue time

Each dot is one CNG driver. The x axis is how long they *could* queue each day before CNG stopped paying (free
conversion); the y axis is how long they report queuing. On the dashed line the two are equal. **Points below the
line are drivers for whom CNG pays.**

Design choices: one colour *and* one marker shape per vehicle type, so the groups can be told apart without relying
on colour (one of the colours is low-contrast on white). Both axes use the same scale and start at 0, so the 45° line really is
"equal". Hover over a point to see the driver's figures.

In [8]:
COLORS = {"ride_hailing": "#2a78d6", "danfo": "#eb6834", "keke": "#1baf7a"}
SYMBOLS = {"ride_hailing": "circle", "danfo": "square", "keke": "diamond"}
LABELS = {"ride_hailing": "Ride-hailing car", "danfo": "Danfo / minibus", "keke": "Keke"}
INK = "#0b0b0b"
INK_MUTED = "#52514e"

# Count drivers below the line, so the title states what the data shows.
n_pays = (cng["daily_queue_hours"] < cng["break_even_daily_queue_hours"]).sum()
axis_max = max(cng["break_even_daily_queue_hours"].max(), cng["daily_queue_hours"].max()) * 1.08

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=[0, axis_max], y=[0, axis_max], mode="lines", name="Break-even (actual = break-even)",
    line=dict(color=INK_MUTED, width=2, dash="dash"), hoverinfo="skip",
))
fig.add_annotation(
    x=axis_max * 0.97, y=axis_max * 0.97, text="Queue = break-even", showarrow=False,
    xanchor="right", yanchor="bottom", font=dict(color=INK_MUTED, size=12),
)
fig.add_annotation(
    x=axis_max * 0.97, y=axis_max * 0.05, text="Below the line: CNG pays", showarrow=False,
    xanchor="right", font=dict(color=INK_MUTED, size=12),
)

for vtype in ["ride_hailing", "danfo", "keke"]:
    d = cng[cng["vehicle_type"] == vtype]
    fig.add_trace(go.Scatter(
        x=d["break_even_daily_queue_hours"], y=d["daily_queue_hours"],
        name=f"{LABELS[vtype]} (n={len(d)})", mode="markers",
        marker=dict(size=11, color=COLORS[vtype], symbol=SYMBOLS[vtype], line=dict(color="white", width=2)),
        customdata=d[["respondent_id", "net_per_hour", "fuel_saving", "net_benefit"]],
        hovertemplate=(
            f"<b>%{{customdata[0]}}</b> · {LABELS[vtype]}<br>"
            "Actual queue: <b>%{y:.2f} h/day</b><br>"
            "Break-even: <b>%{x:.1f} h/day</b><br>"
            "Fuel saving ₦%{customdata[2]:,.0f}/day · earnings ₦%{customdata[1]:,.0f}/h<br>"
            "Net benefit ₦%{customdata[3]:,.0f}/day"
            "<extra></extra>"
        ),
    ))

fig.update_layout(
    title=dict(
        text=f"{n_pays} of {len(cng)} surveyed CNG drivers queue less than their break-even<br>"
             f"<sup>Lagos driver survey (S47), {len(cng)} unflagged CNG users, free conversions (loan cost 0). "
             "Earnings = gross minus CNG spend, per hour.</sup>",
        x=0, xanchor="left",
    ),
    template="plotly_white",
    font=dict(family="Inter, Segoe UI, Arial, sans-serif", size=13, color=INK),
    xaxis=dict(title="Break-even daily queue time (hours)", range=[0, axis_max], zeroline=False,
               gridcolor="#ecebe7", constrain="domain"),
    yaxis=dict(title="Actual daily queue time (hours)", range=[0, axis_max], zeroline=False,
               gridcolor="#ecebe7", scaleanchor="x", scaleratio=1, constrain="domain"),
    legend=dict(orientation="h", yanchor="top", y=-0.12, x=0, xanchor="left"),
    hoverlabel=dict(bgcolor="white", font=dict(color=INK)),
    width=760, height=760, margin=dict(t=100, l=70, r=30, b=120),
)
fig.show()

In [9]:
out_path = FIG_DIR / "survey_break_even.html"
fig.write_html(out_path, include_plotlyjs="cdn")   # cdn: small file, needs internet to render (see notebook 01)
print(f"Saved {out_path.relative_to(PROJECT_ROOT)}")

Saved outputs\figures\survey_break_even.html


## 7. What the results mean

**For most of these drivers, CNG pays, but often not by much.** 9 of the 13 unflagged CNG drivers come out ahead
once the earnings lost in queues are counted. The median driver saves about **₦36,000 a day** on fuel and loses about
**₦21,000** of earnings in queues, leaving a net benefit of about **₦16,700 a day**. Their median break-even is
**2.9 hours** of queuing a day, against a median actual queue of **2.25 hours**: a cushion of under an hour.

**Four drivers lose money: R01 and R11 (danfo), R07 (ride-hailing) and R14 (keke).** They either queue for long
(3 hours a day for R01, R11 and R14) or earn a lot per hour (R07, about ₦23,900), so the earnings they lose in queues
are more than their fuel saving. The keke driver has the smallest fuel saving in the sample (₦5,800 a day), so even a
modest queue wipes it out.

**Queue time, not the kit price, decides the result.** If these drivers had paid the market price of ₦1.3m on a
17.5%, 2-year loan, they would repay about ₦2,100–2,500 per working day. That is small next to their fuel savings
(₦22,000–59,000 a day for all but the keke driver), so the same 9 drivers still come out ahead and the median net
benefit falls only to about ₦14,200 a day. By contrast, one extra hour in the queue costs the median driver about
₦11,200. In notebook 02, loan terms mattered much more. That is because the published figures behind its central
scenario have much lower earnings and fuel savings than these drivers report.

**How the survey compares with the model's assumptions:**
- **Earnings per hour:** the survey median is about ₦11,200 an hour, well above the model's ₦1,000–5,000 range. The
  survey figure has not had owner remittance or app commission taken off, which the model's range assumes, so the two
  are not like-for-like. Even so, the model's range looks too low for this sample.
- **Queuing:** drivers queue for less each time than the model assumes (median 0.75 hours against 4) but refuel more
  often (median 2 a day against 1). Their total daily queue time is 2.25 hours against the model's 4.
- **Working days:** the median driver works 7 days a week (364 days a year) against the model's assumed 312.
- **Fuel saving:** the survey median is 81.7%, a little above the model's central 77.3%.

**Limitations (see `docs/survey_method.md`):**
- **Small, non-random sample.** 13 CNG drivers, found at CNG stations and car parks in Lagos. The keke group is one
  driver. These results describe these drivers, not Lagos drivers in general.
- **Recalled petrol spend.** The fuel saving rests on what drivers remember spending on petrol before converting.
  Some recalled figures are a large share of gross earnings (₦60,000 of ₦78,000 for R05; ₦35,000 of ₦42,000 for
  R12), so the savings may be overstated.
- **Bi-fuel drivers.** 9 of the 13 also run on petrol, and their current petrol spend was not asked. Their real
  fuel saving is smaller than shown here, so their net benefit is overstated.
- **Mostly free conversions.** 12 of the 13 converted through the free programme, and the other's owner paid. The
  market-price scenario is a "what if", not something these drivers experienced. Applying the ₦1.3m car price to the
  keke and danfo drivers is a simplification.
- **"Net" earnings are gross minus fuel only.** Owner remittance and app commission are not deducted. Lower
  take-home pay per hour would make queue time cost less and raise every driver's break-even.